In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
CHUNKS = "/content/drive/MyDrive/trustMed/chunks.jsonl"
chunks = [json.loads(line) for line in open(CHUNKS, encoding="utf-8")]

print("Chunks loaded:", len(chunks))
print(chunks[0]["section"], "|", chunks[0]["title"][:80])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Chunks loaded: 69282
abstract | Efficacy, safety, pharmacokinetics and biomarker findings in patients with HER2-


In [ ]:
!pip install -q sentence-transformers
from sentence_transformers import SentenceTransformer

# Test set: abstracts + conclusions only (short summaries, quick to embed)
test_chunks = [c for c in chunks if c["section"] in ("abstract", "conclusion")]
test_texts = [c["text"] for c in test_chunks]
print("Test chunks:", len(test_texts))

models = {
    "general (MiniLM)": SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda"),
    "medical (PubMedBERT)": SentenceTransformer("NeuML/pubmedbert-base-embeddings", device="cuda"),
}

embeddings = {}
for name, model in models.items():
    print("Embedding with", name, "...")
    embeddings[name] = model.encode(test_texts, batch_size=64,
                                    normalize_embeddings=True, show_progress_bar=True)
print("Done")

Test chunks: 8614


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding with general (MiniLM) ...


Batches:   0%|          | 0/135 [00:00<?, ?it/s]

Embedding with medical (PubMedBERT) ...


Batches:   0%|          | 0/135 [00:00<?, ?it/s]

Done


In [ ]:
print("MiniLM reads up to", models["general (MiniLM)"].max_seq_length, "tokens")

models["general (BGE-small)"] = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cuda")
print("BGE reads up to", models["general (BGE-small)"].max_seq_length, "tokens")

embeddings["general (BGE-small)"] = models["general (BGE-small)"].encode(
    test_texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True)

MiniLM reads up to 256 tokens


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BGE reads up to 512 tokens


Batches:   0%|          | 0/135 [00:00<?, ?it/s]

In [ ]:
import numpy as np

# (question, key term that a relevant result should contain)
questions = [
    ("Does trastuzumab improve survival in HER2-positive breast cancer?", "trastuzumab"),
    ("What are the side effects of tamoxifen?", "tamoxifen"),
    ("Does exercise reduce fatigue in breast cancer survivors?", "exercise"),
    ("Is neoadjuvant chemotherapy effective in triple-negative breast cancer?", "triple-negative"),
    ("Benefits of radiotherapy after breast-conserving surgery", "radiotherapy"),
    ("Does sentinel lymph node biopsy cause less arm swelling than axillary dissection?", "lymph"),
    ("Are aromatase inhibitors better than tamoxifen for postmenopausal women?", "aromatase"),
    ("Do CDK4/6 inhibitors improve progression-free survival?", "cdk4/6"),
    ("Does mammography screening reduce breast cancer deaths?", "screening"),
    ("Risk of heart damage from anthracycline chemotherapy", "cardi"),
    ("Does pembrolizumab work in triple-negative breast cancer?", "pembrolizumab"),
    ("PARP inhibitors for patients with BRCA mutations", "parp"),
    ("Is hypofractionated radiotherapy as good as conventional radiotherapy?", "hypofraction"),
    ("Does acupuncture help with hot flashes?", "acupuncture"),
    ("Can scalp cooling prevent hair loss during chemotherapy?", "scalp"),
    ("Depression and anxiety in women with breast cancer", "depress"),
    ("Is obesity linked to breast cancer recurrence?", "obes"),
    ("Does pathologic complete response predict survival?", "pathologic complete response"),
    ("Using circulating tumour DNA to monitor treatment response", "ctdna"),
    ("Complications of breast reconstruction after mastectomy", "reconstruction"),
]

TOP_K = 5
scores = {}
for name, model in models.items():
    q_emb = model.encode([q for q, _ in questions], normalize_embeddings=True)
    sims = q_emb @ embeddings[name].T            # similarity of every question to every chunk
    hits = []
    for i, (q, term) in enumerate(questions):
        top = np.argsort(-sims[i])[:TOP_K]
        hits.append(sum(term in test_texts[j].lower() for j in top) / TOP_K)
    scores[name] = hits
    print(f"{name}: {np.mean(hits):.0%} of top-{TOP_K} results contain the key term")

print("\nPer question (general vs medical):")
names = list(models)
for i, (q, _) in enumerate(questions):
    print(f"{scores[names[0]][i]:.0%}  vs  {scores[names[1]][i]:.0%}   {q}")

general (MiniLM): 92% of top-5 results contain the key term
medical (PubMedBERT): 71% of top-5 results contain the key term
general (BGE-small): 94% of top-5 results contain the key term

Per question (general vs medical):
100%  vs  100%   Does trastuzumab improve survival in HER2-positive breast cancer?
100%  vs  40%   What are the side effects of tamoxifen?
100%  vs  100%   Does exercise reduce fatigue in breast cancer survivors?
80%  vs  40%   Is neoadjuvant chemotherapy effective in triple-negative breast cancer?
100%  vs  60%   Benefits of radiotherapy after breast-conserving surgery
100%  vs  80%   Does sentinel lymph node biopsy cause less arm swelling than axillary dissection?
80%  vs  40%   Are aromatase inhibitors better than tamoxifen for postmenopausal women?
100%  vs  100%   Do CDK4/6 inhibitors improve progression-free survival?
100%  vs  100%   Does mammography screening reduce breast cancer deaths?
100%  vs  100%   Risk of heart damage from anthracycline chemotherapy
10

In [ ]:
def show(question, n=3):
    print("Q:", question)
    for name, model in models.items():
        q = model.encode([question], normalize_embeddings=True)
        top = np.argsort(-(q @ embeddings[name].T)[0])[:n]
        print(f"\n  --- {name} ---")
        for j in top:
            print("  •", test_chunks[j]["title"][:90])
    print("\n" + "=" * 90)

show("Does pembrolizumab work in triple-negative breast cancer?")
show("What are the side effects of tamoxifen?")

Q: Does pembrolizumab work in triple-negative breast cancer?

  --- general (MiniLM) ---
  • Pembrolizumab and chemotherapy in high-risk, early-stage, ER+/HER2- breast cancer: a rando
  • Pembrolizumab plus chemotherapy in Japanese patients with triple-negative breast cancer: R
  • Pembrolizumab Plus Chemotherapy Followed by Pembrolizumab in Patients With Early Triple-Ne

  --- medical (PubMedBERT) ---
  • Targeted Treatment of Metastatic Triple-Negative Breast Cancer: A Systematic Review.
  • Atezolizumab in early triple-negative breast cancer: the randomized phase 3 NSABP B-59/Gep
  • Neoadjuvant therapy in triple-negative breast cancer: A systematic review and network meta

  --- general (BGE-small) ---
  • Pembrolizumab and chemotherapy in high-risk, early-stage, ER+/HER2- breast cancer: a rando
  • Neoadjuvant immunotherapy and chemotherapy regimens for the treatment of high-risk, early-
  • Molecular determinants of response to neoadjuvant pembrolizumab plus chemotherapy in patie

In [ ]:
!pip install -q chromadb
import chromadb

model = models["general (BGE-small)"]

# 1) Embed every chunk
all_emb = model.encode([c["text"] for c in chunks], batch_size=128,
                       normalize_embeddings=True, show_progress_bar=True)

# 2) Store in ChromaDB
client = chromadb.PersistentClient(path="/content/chroma_db")
col = client.get_or_create_collection("trustmed_chunks", metadata={"hnsw:space": "cosine"})

def meta(c):
    # ChromaDB only accepts simple values (text, numbers, True/False), so lists become text
    return {
        "pmid": c["pmid"], "section": c["section"], "title": c["title"],
        "journal": c["journal"] or "", "year": c["year"] or 0,
        "doi": c["doi"] or "", "pmcid": c["pmcid"] or "",
        "authors": ", ".join(c["authors"]),
        "study_type": c["study_type"], "evidence_level": c["evidence_level"],
        "retraction_status": c["retraction_status"], "retracted": bool(c["retracted"]),
    }

BATCH = 5000
for i in range(0, len(chunks), BATCH):
    part = chunks[i:i + BATCH]
    col.upsert(ids=[c["chunk_id"] for c in part],
               documents=[c["text"] for c in part],
               embeddings=all_emb[i:i + BATCH].tolist(),
               metadatas=[meta(c) for c in part])
    print(f"Stored {min(i + BATCH, len(chunks))}/{len(chunks)}")

print("Chunks in database:", col.count())

# 3) Save a copy to Google Drive
!cp -r /content/chroma_db /content/drive/MyDrive/trustMed/
print("Saved to Google Drive: trustMed/chroma_db")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.8/80.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/13

Batches:   0%|          | 0/542 [00:00<?, ?it/s]

Stored 5000/69282
Stored 10000/69282
Stored 15000/69282
Stored 20000/69282
Stored 25000/69282
Stored 30000/69282
Stored 35000/69282
Stored 40000/69282
Stored 45000/69282
Stored 50000/69282
Stored 55000/69282
Stored 60000/69282
Stored 65000/69282
Stored 69282/69282
Chunks in database: 69282
Saved to Google Drive: trustMed/chroma_db
